In [1]:
import os
import json
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ================= CẤU HÌNH =================

# 1. Model ID (Model T5 Seq2Seq)
MODEL_ID = "phuc-hoang1208/finetuned-vit5base-textsplitting"

# 2. File kết quả
OUTPUT_FILE = "vit5_atoss_results.jsonl"

# ================= THIẾT LẬP MODEL (SEQ2SEQ) =================

print(f"⏳ Đang tải Model {MODEL_ID}...")

# Kiểm tra GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Thiết bị sử dụng: {device.upper()}")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    # Model ViT5-base nhẹ nên load thẳng float32 hoặc float16, không cần 4-bit
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device)
    print("✅ Model đã tải thành công!")
except Exception as e:
    print(f"❌ Lỗi tải model: {e}")
    raise e

# ================= HÀM INFERENCE (SEQ2SEQ) =================

def call_vit5_model(sentence, gold_label):
    """
    Với model Fine-tuned, ta đưa input trực tiếp thay vì prompt dài dòng.
    Định dạng input phụ thuộc vào lúc bạn train model. 
    Dưới đây là giả định format chuẩn cho ATOSS (Sentence + Quadruplets guide).
    """
    
    # Tạo input string. 
    # Nếu model được train chỉ với câu gốc -> bỏ phần gold_label đi.
    # Nhưng theo bài báo ATOSS (Specific), input cần cả Quadruplets để định hướng.
    input_text = sentence
    
    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt", max_length=256, truncation=True).to(device)

    try:
        with torch.no_grad():
            # Sinh văn bản
            outputs = model.generate(
                inputs.input_ids,
                max_length=512,
                num_beams=5,            # Dùng Beam Search để tìm kết quả tốt nhất
                num_return_sequences=3, # Sinh ra 3 biến thể khác nhau (Top 3 beams)
                early_stopping=True,
                no_repeat_ngram_size=2  # Tránh lặp từ
            )
        
        # Giải mã kết quả (Decode)
        decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        # Làm sạch kết quả
        variations = [v.strip() for v in decoded_outputs if v.strip()]
        
        # Trả về danh sách unique (loại bỏ trùng lặp nếu có)
        return list(set(variations))

    except Exception as e:
        print(f"⚠️ Lỗi Inference: {e}")
        return []

# ================= MAIN LOGIC =================

def main():
    print("⏳ Đang tải dataset...")
    try:
        ds = load_dataset("vohuutridung/3190-data")
        data_source = ds["train"]
        print(f"✅ Đã tải {len(data_source)} dòng.")
    except Exception as e:
        print(f"❌ Lỗi tải data: {e}")
        return

    # Kiểm tra resume
    processed_ids = set()
    if os.path.exists(OUTPUT_FILE):
        print("📂 Tìm thấy file cũ, đang kiểm tra resume...")
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    processed_ids.add(data.get('original', ''))
                except: pass
        print(f"⏭️ Đã xử lý {len(processed_ids)} dòng. Sẽ bỏ qua.")

    print(f"🚀 Bắt đầu xử lý (Model: {MODEL_ID})...")

    with open(OUTPUT_FILE, 'a', encoding='utf-8', buffering=1) as f_out:
        
        for idx, item in tqdm(enumerate(data_source), total=len(data_source)):
            try:
                # 1. Lấy dữ liệu
                keys = list(item.keys())
                text_col = next((k for k in keys if k in ['text', 'sentence']), keys[0])
                label_col = next((k for k in keys if k in ['labels', 'label']), keys[1])
                
                original_sent = str(item[text_col]).strip()
                gold_label = str(item[label_col])
                
                if "####" in original_sent:
                    original_sent, gold_label = original_sent.split("####")

                # 2. Skip nếu đã làm
                if original_sent in processed_ids:
                    continue

                # 3. Gọi Model ViT5
                splits = call_vit5_model(original_sent, gold_label)
                
                if splits:
                    result_item = {
                        "id": idx,
                        "original": original_sent,
                        "gold_label": gold_label,
                        "variations": splits
                    }
                    
                    f_out.write(json.dumps(result_item, ensure_ascii=False) + "\n")
                    f_out.flush()

            except Exception as ex:
                print(f"❌ Lỗi dòng {idx}: {ex}")
                continue

    print(f"\n🎉 HOÀN TẤT! File kết quả: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

⏳ Đang tải Model phuc-hoang1208/finetuned-vit5base-textsplitting...
⚙️ Thiết bị sử dụng: CUDA


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Generating validation split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/597 [00:00<?, ? examples/s]

✅ Đã tải 10000 dòng.
🚀 Bắt đầu xử lý (Model: phuc-hoang1208/finetuned-vit5base-textsplitting)...


100%|██████████| 10000/10000 [3:02:21<00:00,  1.09s/it]


🎉 HOÀN TẤT! File kết quả: vit5_atoss_results.jsonl
